In [1]:
import numpy as np
import random

In [2]:
class SimpleGridWorld:
    """
    A 4x4 GridWorld environment.
    - Start: (0, 0)
    - Goal:  (3, 3) -> Reward: +10
    - Trap:  (1, 1) -> Reward: -10
    - Step penalty: -1 (encourages shortest path)
    """
    def __init__(self):
        self.rows = 4
        self.cols = 4
        self.start = (0, 0)
        self.goal = (3, 3)
        self.trap = (1, 1)
        self.state = self.start
        # Actions: 0=Up, 1=Down, 2=Left, 3=Right
        self.actions = [0, 1, 2, 3]

    def reset(self):
        self.state = self.start
        return self._state_to_idx(self.state)

    def _state_to_idx(self, state):
        return state[0] * self.cols + state[1]

    def step(self, action):
        r, c = self.state
        if action == 0:   # Up
            r = max(0, r - 1)
        elif action == 1: # Down
            r = min(self.rows - 1, r + 1)
        elif action == 2: # Left
            c = max(0, c - 1)
        elif action == 3: # Right
            c = min(self.cols - 1, c + 1)

        self.state = (r, c)
        next_idx = self._state_to_idx(self.state)

        if self.state == self.goal:
            return next_idx, 10, True
        elif self.state == self.trap:
            return next_idx, -10, True
        else:
            return next_idx, -1, False

In [3]:
def choose_action(state, q_table, epsilon, num_actions):
    """Epsilon-greedy action selection."""
    if random.uniform(0, 1) < epsilon:
        return random.randint(0, num_actions - 1)  # Explore
    return np.argmax(q_table[state])               # Exploit

In [4]:
def train_q_learning(env, episodes=500, alpha=0.1, gamma=0.95, epsilon=0.1):
    """Off-policy TD control: Q-learning."""
    q_table = np.zeros((env.rows * env.cols, len(env.actions)))

    for _ in range(episodes):
        state = env.reset()
        done = False

        while not done:
            action = choose_action(state, q_table, epsilon, len(env.actions))
            next_state, reward, done = env.step(action)

            # Q-learning update uses the MAXIMUM Q-value of the next state
            best_next_q = np.max(q_table[next_state])
            td_target = reward + gamma * best_next_q * (not done)
            q_table[state, action] += alpha * (td_target - q_table[state, action])

            state = next_state

    return q_table

In [5]:
def train_sarsa(env, episodes=500, alpha=0.1, gamma=0.95, epsilon=0.1):
    """On-policy TD control: SARSA."""
    q_table = np.zeros((env.rows * env.cols, len(env.actions)))

    for _ in range(episodes):
        state = env.reset()
        action = choose_action(state, q_table, epsilon, len(env.actions))
        done = False

        while not done:
            next_state, reward, done = env.step(action)
            next_action = choose_action(next_state, q_table, epsilon, len(env.actions))

            # SARSA update uses the ACTUAL next action chosen by the policy
            next_q = q_table[next_state, next_action]
            td_target = reward + gamma * next_q * (not done)
            q_table[state, action] += alpha * (td_target - q_table[state, action])

            state = next_state
            action = next_action

    return q_table

In [6]:
def print_policy(q_table, env):
    """Visualizes the learned optimal action for each grid cell."""
    action_symbols = {0: '↑', 1: '↓', 2: '←', 3: '→'}
    for r in range(env.rows):
        row_str = []
        for c in range(env.cols):
            if (r, c) == env.goal:
                row_str.append('G')
            elif (r, c) == env.trap:
                row_str.append('X')
            else:
                state_idx = r * env.cols + c
                best_action = np.argmax(q_table[state_idx])
                row_str.append(action_symbols[best_action])
        print(" ".join(row_str))


In [ ]:

if __name__ == "__main__":
    random.seed(42)
    np.random.seed(42)

    env = SimpleGridWorld()

    print("Training Q-Learning...")
    q_learning_table = train_q_learning(env)
    print("Learned Policy (Q-Learning):")
    print_policy(q_learning_table, env)

    print("\nTraining SARSA...")
    sarsa_table = train_sarsa(env)
    print("Learned Policy (SARSA):")
    print_policy(sarsa_table, env)     

Training Q-Learning...
Learned Policy (Q-Learning):
→ → ↓ ↓
↓ X ↓ ↓
→ → → ↓
↑ → → G

Training SARSA...
Learned Policy (SARSA):
→ → ↓ ↓
↓ X ↓ ↓
→ → ↓ ↓
→ → → G
